# 🩺 糖尿病患病風險評估 Gradio Web UI 實作教學

> 對象：修習機器學習模型部署與 Web UI 開發的學員
> 對應程式碼：`app.py` 第 109~171 行 (`Gradio UI` 區塊)
>
> 本 Notebook 帶你拆解 Gradio 互動式醫療儀表板：從輸入滑桿、事件處理器、風險標籤、到使用 Matplotlib 動態繪製臨床危險因子勝算比 (Odds Ratio) 長條圖。

## 🎯 學習目標

- [ ] 掌握 Gradio `Blocks` 雙欄排版架構（左側病患指標輸入、右側診斷結果與圖表）
- [ ] 能將邏輯迴歸模型與 `StandardScaler` 包裝進 Gradio 事件回呼函數 (`gradio_predict`)
- [ ] 理解如何在 Gradio 中使用 Matplotlib 動態繪製水平長條圖解析勝算比 (Odds Ratio)
- [ ] 了解多輸出綁定 (`outputs=[out_prob, out_badge, out_advice, out_plot]`)
- [ ] 掌握如何將 Gradio App 掛載至 FastAPI (`gr.mount_gradio_app`)

In [ ]:
# ============================================
# 0. 載入套件與模型
# ============================================
import os, joblib
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr

MODEL_PATH = "diabetes_model.joblib"
if not os.path.exists(MODEL_PATH):
    from train_save import train_and_save_model
    train_and_save_model(model_path=MODEL_PATH)

meta = joblib.load(MODEL_PATH)
model = meta["model"]
scaler = meta["scaler"]
odds_ratios = meta["odds_ratios"]
print("✅ 模型載入完成！")

---
## Part 1：定義 Gradio 推論與繪圖核心函數 (`gradio_predict`)

函數接收 `(age, weight, blood_sugar, gender)`，計算患病機率、判斷風險等級，並繪製危險因子勝算比長條圖。

In [ ]:
def gradio_predict(age, weight, blood_sugar, gender):
    gender_num = 1 if gender == "男生" else 0
    raw_features = np.array([[age, weight, blood_sugar, gender_num]])
    scaled_features = scaler.transform(raw_features)
    prob = float(model.predict_proba(scaled_features)[0, 1])
    
    prob_percent = f"{prob * 100:.1f}%"
    if prob >= 0.7:
        badge = "🔴 【極高風險】"
        advice = "臨床建議：空腹血糖值或年齡權重偏高，建議安排進一步 OGTT 耐糖試驗與新陳代謝專科追蹤。"
    elif prob >= 0.4:
        badge = "🟡 【中度警示】"
        advice = "臨床建議：數值接近前期臨界值，建議每半年複驗空腹血糖並落實低 GI 飲食。"
    else:
        badge = "🟢 【低風險健康】"
        advice = "臨床建議：各項指標良好，持續維持均衡營養與定期健檢。"
        
    # 繪製勝算比圖
    fig, ax = plt.subplots(figsize=(6, 3))
    features = list(odds_ratios.keys())
    ors = list(odds_ratios.values())
    bars = ax.barh(features, ors, color="teal")
    ax.axvline(1.0, color="red", linestyle="--", alpha=0.7, label="基準 (OR=1.0)")
    ax.set_title("臨床危險因子勝算比 (Odds Ratio)", fontsize=11)
    ax.set_xlabel("勝算比 (OR)", fontsize=10)
    ax.legend(fontsize=9)
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.05, bar.get_y() + bar.get_height()/2, f"{w:.2f}", va="center", fontsize=9)
    plt.tight_layout()
    
    return prob_percent, badge, advice, fig

# 測試執行一次
p, b, a, f = gradio_predict(45, 75, 126, "男生")
print(f"機率: {p}, 標籤: {b}")

---
## Part 2：建構 Gradio Blocks Web 介面

In [ ]:
with gr.Blocks(title="糖尿病患病風險評估系統", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 智慧醫療：糖尿病患病風險即時評估系統")
    gr.Markdown("本系統採用臨床可解釋性極高的 **邏輯迴歸（Logistic Regression）** 模型，提供即時患病風險機率預估與危險因子勝算比（Odds Ratio）剖析。")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📋 病患臨床指標輸入")
            in_age = gr.Slider(minimum=10, maximum=100, value=45, step=1, label="年齡 (歲)")
            in_weight = gr.Slider(minimum=30, maximum=160, value=75, step=0.5, label="體重 (公斤)")
            in_sugar = gr.Slider(minimum=50, maximum=250, value=115, step=1, label="空腹血糖 (mg/dL)")
            in_gender = gr.Radio(choices=["男生", "女生"], value="男生", label="生理性別")
            btn_predict = gr.Button("開始臨床評估", variant="primary")
            
        with gr.Column(scale=1):
            gr.Markdown("### 📊 評估診斷與分析")
            out_prob = gr.Label(label="預估患病機率")
            out_badge = gr.Textbox(label="風險等級", interactive=False)
            out_advice = gr.Textbox(label="臨床處置建議", interactive=False, lines=3)
            out_plot = gr.Plot(label="危險因子勝算比分析")
            
    btn_predict.click(
        fn=gradio_predict,
        inputs=[in_age, in_weight, in_sugar, in_gender],
        outputs=[out_prob, out_badge, out_advice, out_plot]
    )

print("✅ Gradio Blocks 介面定義完成！可執行 demo.launch() 啟動測試。")